# 1. Local activity recognition with X-CLIP

Rank editable activity descriptions for local video clips using the pinned, pretrained X-CLIP model. No training, external tracking, or network inference is used. Keep source videos local; this notebook does not extract frame files.

**Scores are relative to the supplied candidates, not calibrated confidence.** Mixed activities produce a ranking of candidates, not independent activity labels. ‘Other activities’ is an ordinary text candidate, not a trained rejection mechanism.

Run from `notebooks/` using the uv environment. The empty configuration runs diagnostics and skips model loading and video processing. Edit the tagged input cell, then restart and run all cells. Use distinct video filename stems because result IDs and reference labels use those stems.

In [ ]:
# Import the local runtime first: it configures offline mode before ML imports.
from activity_recognition import (
    DEFAULT_PROMPT_TEMPLATE, NUM_FRAMES, SampledVideo,
    device_info, get_recognizer, prepare_candidates, sample_video,
)

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


In [ ]:
VIDEO_PATHS: list[str] = []
REFERENCE_LABELS: dict[str, str] = {}
ACTIVITIES = [
    "Dancing", "Singing", "Playing sports", "Playing games", "Running",
    "Walking", "Sitting and listening", "Drawing or painting", "Other activities",
]
DEVICE = "auto"
REPEATS = 3
EXPORT_RESULTS = False
OUTPUT_DIR = Path("results")


## 2. Runtime diagnostics

Diagnostics always run. Automatic selection uses CUDA if available, otherwise CPU; `DEVICE = 'cpu'` forces CPU. CUDA uses float16, CPU float32. The actual device and dtype are also recorded with each result. GPU fields are not applicable on CPU.

In [ ]:
runtime_info = device_info()
display(pd.DataFrame([runtime_info]))
print(f'Requested device: {DEVICE}; native frames per classification: {NUM_FRAMES}')
if not VIDEO_PATHS:
    print('Set VIDEO_PATHS to local video paths in the input-configuration cell, then restart and run all.')
    print('Model loading, previews, classification, and experiments are skipped.')


### Separate, explicit model download

Acquire the pinned weights once from a terminal with network access, from `notebooks/`:

```bash
uv run --locked python download_model.py
```

This command is not run by the notebook. Initialization uses only the pinned local cache; a missing or incomplete cache produces an actionable error.

## 3. Model initialization timing

Reuse one recognizer throughout. Its initialization time includes local model loading and device placement, and is separate from per-clip latency. CUDA failures do not silently retry on CPU.

In [ ]:
recognizer = None
model_load_time_seconds = None
if VIDEO_PATHS:
    recognizer = get_recognizer(DEVICE)
    model_load_time_seconds = recognizer.load_time_seconds
    display(pd.DataFrame([{
        'model_load_time_seconds': model_load_time_seconds,
        'device': str(recognizer.device),
        'dtype': str(recognizer.dtype),
        'model_revision': recognizer.model_revision,
    }]))


## 4. Editable clips, categories, and prompts

Edit local paths, category names, device, repeats, and export settings in the input cell above. Edit the experiment definitions below to compare wording or candidate sets. Description mappings must exactly cover the original labels and supply full prompts. All runs keep the checkpoint's native 32 frames.

Optional `REFERENCE_LABELS` maps video stems to canonical keys, for example `{'clip_01': 'playing_sports'}`. These are reference annotations you supply, not labels inferred from filenames.

In [ ]:
DESCRIPTIONS = {
    'Dancing': 'A group of teenagers dancing to music.',
    'Singing': 'People singing together in a group.',
    'Playing sports': 'People actively playing a sport together.',
    'Playing games': 'People playing a board game or a group game.',
    'Running': 'People running across an area.',
    'Walking': 'People walking through an area.',
    'Sitting and listening': 'People sitting and listening to someone speaking.',
    'Drawing or painting': 'People drawing pictures or painting artwork.',
    'Other activities': 'People doing an activity other than the listed activities.',
}
EXPERIMENTS = [
    {
        'name': 'short_labels', 'activities': ACTIVITIES,
        'prompt_template': '{activity}', 'descriptions': None,
    },
    {
        'name': 'descriptive', 'activities': ACTIVITIES,
        'prompt_template': DEFAULT_PROMPT_TEMPLATE, 'descriptions': DESCRIPTIONS,
    },
    {
        'name': 'similar_motion_candidates',
        'activities': ['Dancing', 'Running', 'Walking', 'Playing sports'],
        'prompt_template': DEFAULT_PROMPT_TEMPLATE, 'descriptions': None,
    },
]


## 5. Sampled frame grid and single-clip classification

Preview up to eight uniformly selected positions from the same 32-frame sampling scheme used by the runtime. Images are RGB and remain in memory. Frame index divided by FPS is an approximate timestamp, especially for variable-frame-rate video. Short clips can repeat frames. The model resizes and center-crops frames, which can cut off activities at the edges.

The single-clip result below uses the default prompt template. Its timing may include first-call overhead; warmed measurements are collected in section 8. Preprocessing includes decoding and processor work; inference includes device transfer and completed forward computation. Total excludes model loading. CUDA peaks include the resident model; reserved memory is allocator reservation, not just live tensors.

In [ ]:
def show_sampled_frames(sampled: SampledVideo, *, max_frames: int = 8):
    if isinstance(max_frames, bool) or not isinstance(max_frames, int) or max_frames < 1:
        raise ValueError('max_frames must be a positive integer')
    count = min(8, max_frames, len(sampled.frames))
    if count == 0:
        return
    positions = np.linspace(0, len(sampled.frames) - 1, count, dtype=int)
    columns = min(4, count)
    rows = (count + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 3 * rows), squeeze=False)
    for axis in axes.flat:
        axis.axis('off')
    for axis, position in zip(axes.flat, positions):
        frame_index = sampled.indices[position]
        axis.imshow(sampled.frames[position])
        axis.set_title(f'Frame {frame_index} | ~{frame_index / sampled.fps:.2f} s')
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def show_result(result: dict):
    ranked = sorted(result['scores'].items(), key=lambda item: item[1])
    fig, axis = plt.subplots(figsize=(9, max(3, 0.45 * len(ranked))))
    axis.barh([result['labels'][key] for key, _ in ranked], [score for _, score in ranked])
    axis.set_xlim(0, 1)
    axis.set_xlabel('Relative candidate score (not calibrated confidence)')
    axis.set_title(f"{result['video_id']}: {result['labels'][result['activity']]}")
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    print(f"Prediction: {result['labels'][result['activity']]} ({result['activity']})")
    display(pd.DataFrame([{key: result[key] for key in (
        'device', 'dtype', 'num_frames', 'unique_sampled_frames',
        'preprocessing_time_seconds', 'inference_time_seconds', 'total_time_seconds',
    )}]))
    memory = {key: result[key] if result[key] is not None else 'not applicable (CPU)'
              for key in ('peak_gpu_memory_allocated_mib', 'peak_gpu_memory_reserved_mib')}
    display(pd.DataFrame([memory]))


In [ ]:
single_result = None
if VIDEO_PATHS:
    try:
        sampled = sample_video(VIDEO_PATHS[0], num_frames=NUM_FRAMES)
        show_sampled_frames(sampled)
        single_resultdata = recognizer.classify(
            VIDEO_PATHS[0], ACTIVITIES, prompt_template=DEFAULT_PROMPT_TEMPLATE,
        )
        show_result(single_result)
    except (OSError, ValueError, RuntimeError) as exc:
        print(f'Single-clip preview/classification failed: {type(exc).__name__}: {exc}')


## 6. Raw labels versus full descriptions

Inspect the exact prompts before comparing predictions. `short_labels` and `descriptive` use the same candidates, so the comparison isolates wording. Full descriptions replace the template rather than being inserted into it. Invalid experiment values appear in a separate error table while valid variants continue. Section 8 displays their measured results side by side. Prompt details remain in every result record.

In [ ]:
prompt_rows = []
prompt_error_rows = []
if VIDEO_PATHS:
    for experiment in EXPERIMENTS:
        name = experiment['name']
        try:
            candidates = prepare_candidates(
                experiment['activities'], prompt_template=experiment['prompt_template'],
                descriptions=experiment['descriptions'],
            )
        except ValueError as exc:
            prompt_error_rows.append({
                'experiment': name, 'error_type': type(exc).__name__, 'message': str(exc),
            })
            continue
        prompt_rows.extend({
            'experiment': name, 'key': candidate.key,
            'label': candidate.label, 'prompt': candidate.prompt,
        } for candidate in candidates)
prompt_comparison_df = pd.DataFrame(prompt_rows, columns=['experiment', 'key', 'label', 'prompt'])
prompt_errors_df = pd.DataFrame(prompt_error_rows, columns=['experiment', 'error_type', 'message'])
if VIDEO_PATHS:
    display(prompt_comparison_df)
    if not prompt_errors_df.empty:
        display(prompt_errors_df)


## 7. Candidate-set and overlapping-activity experiments

The motion subset compares Dancing, Running, Walking, and Playing sports, which can overlap visually. Change `EXPERIMENTS` to explore alternatives. Adding or removing categories changes softmax normalization, so scores from different candidate sets are not directly comparable probabilities. A high-ranked activity can coexist with others in the clip.

The table lists valid experiments' candidate keys; a separate error table identifies invalid experiment values. Compare rankings and top choices in section 8; do not interpret the ordinary ‘Other activities’ prompt as an unknown-activity detector.

In [ ]:
candidate_rows = []
candidate_error_rows = []
if VIDEO_PATHS:
    for experiment in EXPERIMENTS:
        name = experiment['name']
        try:
            candidates = prepare_candidates(
                experiment['activities'], prompt_template=experiment['prompt_template'],
                descriptions=experiment['descriptions'],
            )
        except ValueError as exc:
            candidate_error_rows.append({
                'experiment': name, 'error_type': type(exc).__name__, 'message': str(exc),
            })
            continue
        candidate_rows.append({
            'experiment': name, 'candidate_count': len(candidates),
            'candidate_keys': [candidate.key for candidate in candidates],
        })
candidate_sets_df = pd.DataFrame(candidate_rows, columns=[
    'experiment', 'candidate_count', 'candidate_keys',
])
candidate_errors_df = pd.DataFrame(candidate_error_rows, columns=[
    'experiment', 'error_type', 'message',
])
if VIDEO_PATHS:
    display(candidate_sets_df)
    if not candidate_errors_df.empty:
        display(candidate_errors_df)


## 8. Sequential batch experiments, warmed latency, and references

Run one clip at a time with the existing recognizer. Each video/experiment pair gets one labeled warmup call, excluded from records and timing summaries, followed by `REPEATS` measured calls. Expected file, input-value, and runtime failures become separate error rows and processing continues with the next pair. Programming errors such as missing dictionary keys or incorrect types propagate.

The flattened DataFrame preserves per-candidate `scores.*` columns and all runtime metadata. Latency summaries show median/min/max across measured repeats per clip and experiment. GPU memory summaries show peak allocated/reserved MiB; CPU values are not applicable. Reference comparisons include only annotated video stems, report distinct labeled clips, and do not treat repeats as independent labeled examples.

In [ ]:
def run_experiments(
    video_paths: list[str], experiments: list[dict], *, repeats: int = 1,
) -> tuple[list[dict], pd.DataFrame, pd.DataFrame]:
    if isinstance(repeats, bool) or not isinstance(repeats, int) or repeats < 1:
        raise ValueError('repeats must be a positive integer')
    if video_paths and experiments and recognizer is None:
        raise RuntimeError('Initialize the recognizer before running configured experiments')
    successful_records = []
    error_rows = []
    for video_path in video_paths:
        video_id = Path(video_path).stem
        for experiment in experiments:
            name = experiment['name']
            options = {
                'prompt_template': experiment['prompt_template'],
                'descriptions': experiment['descriptions'],
                'video_id': video_id,
            }
            stage = 'warmup'
            repeat_index = None
            try:
                candidates = prepare_candidates(
                    experiment['activities'],
                    prompt_template=options['prompt_template'],
                    descriptions=options['descriptions'],
                )
                strategy = ('descriptions' if options['descriptions'] is not None else
                            'raw_labels' if options['prompt_template'] == '{activity}' else 'template')
                print(f'Warmup (excluded): {video_id} / {name}')
                recognizer.classify(video_path, experiment['activities'], **options)
                stage = 'measured'
                for repeat_index in range(repeats):
                    result = recognizer.classify(video_path, experiment['activities'], **options)
                    successful_records.append({
                        **result, 'experiment': name, 'prompt_strategy': strategy,
                        'candidate_keys': [candidate.key for candidate in candidates],
                        'repeat_index': repeat_index,
                    })
            except (OSError, ValueError, RuntimeError) as exc:
                error_rows.append({
                    'video_id': video_id, 'experiment': name, 'stage': stage,
                    'repeat_index': repeat_index,
                    'error_type': type(exc).__name__, 'message': str(exc),
                })
    flattened = pd.json_normalize(successful_records, sep='.')
    errors = pd.DataFrame(error_rows, columns=[
        'video_id', 'experiment', 'stage', 'repeat_index', 'error_type', 'message',
    ])
    return successful_records, flattened, errors


In [ ]:
records = []
results_df = pd.DataFrame()
errors_df = pd.DataFrame()
if VIDEO_PATHS:
    records, results_df, errors_df = run_experiments(VIDEO_PATHS, EXPERIMENTS, repeats=REPEATS)
    display(results_df)
    if not errors_df.empty:
        display(errors_df)

latency_summary = pd.DataFrame()
memory_summary = pd.DataFrame()
ranking_df = pd.DataFrame()
if not results_df.empty:
    groups = results_df.groupby(['video_id', 'experiment'], sort=False)
    latency_summary = groups[[
        'preprocessing_time_seconds', 'inference_time_seconds', 'total_time_seconds',
    ]].agg(['median', 'min', 'max'])
    display(latency_summary)
    memory_summary = groups[[
        'peak_gpu_memory_allocated_mib', 'peak_gpu_memory_reserved_mib',
    ]].max()
    display(memory_summary.astype(object).where(memory_summary.notna(), 'not applicable (CPU)'))
    ranking_rows = []
    for result in records:
        for rank, (key, score) in enumerate(
            sorted(result['scores'].items(), key=lambda item: item[1], reverse=True), start=1,
        ):
            ranking_rows.append({
                'video_id': result['video_id'], 'experiment': result['experiment'],
                'repeat_index': result['repeat_index'], 'rank': rank,
                'activity_key': key, 'label': result['labels'][key], 'relative_score': score,
            })
    ranking_df = pd.DataFrame(ranking_rows)
    display(ranking_df)
    print('Top choices by clip and experiment (first measured repeat):')
    display(results_df[results_df['repeat_index'].eq(0)].pivot(
        index='video_id', columns='experiment', values='activity',
    ))


In [ ]:
reference_comparison = pd.DataFrame()
reference_summary = pd.DataFrame()
if not results_df.empty:
    reference_comparison = results_df.loc[
        results_df['video_id'].isin(REFERENCE_LABELS),
        ['video_id', 'experiment', 'repeat_index', 'activity', 'candidate_keys'],
    ].copy()
    if not reference_comparison.empty:
        reference_comparison['reference_key'] = reference_comparison['video_id'].map(REFERENCE_LABELS)
        reference_comparison['reference_in_candidates'] = [
            reference in keys for reference, keys in zip(
                reference_comparison['reference_key'], reference_comparison['candidate_keys'],
            )
        ]
        reference_comparison['top_match'] = (
            reference_comparison['activity'] == reference_comparison['reference_key']
        )
        display(reference_comparison)
        print(f"Unique labeled clips compared: {reference_comparison['video_id'].nunique()}")
        first_repeats = reference_comparison[reference_comparison['repeat_index'].eq(0)]
        reference_summary = first_repeats.groupby('experiment').agg(
            unique_labeled_clips=('video_id', 'nunique'),
            top_matches_first_repeat=('top_match', 'sum'),
            references_in_candidates=('reference_in_candidates', 'sum'),
        )
        display(reference_summary)
    else:
        print('No reference-labeled clips among successful results; no reference comparison computed.')


## 9. Explicit local exports

Only the cell below writes derived files, and only when `EXPORT_RESULTS = True` and successful records exist. Each explicit export overwrites JSON records, flattened CSV results, and the errors CSV for the current run. The errors CSV is header-only when no failures occurred. Frame files are never written. Exported metadata includes prompts, model revision, sampled indices, relative scores, logits, timings, and device information.

**Jupyter can persist preview frames and metadata in saved cell outputs.** Clear outputs before committing or sharing the notebook. From `notebooks/`:

```bash
uv run --locked jupyter nbconvert --ClearOutputPreprocessor.enabled=True --inplace activity_recognition.ipynb
```

In [ ]:
if EXPORT_RESULTS and records:
    output_dir = OUTPUT_DIR
    serialized_records = json.dumps(records, indent=2, allow_nan=False)
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / 'activity_results.json').write_text(serialized_records, encoding='utf-8')
    results_df.to_csv(output_dir / 'activity_results.csv', index=False)
    errors_df.to_csv(output_dir / 'activity_errors.csv', index=False)
    print(f'Exported results to {output_dir.resolve()}')
else:
    print('No exports: enable EXPORT_RESULTS and collect successful records to write results.')
